# Mi proyecto: Delivery Insights (ETL + Business Intelligence)

## EXTRACT

In [ ]:
import os

# Ruta de la carpeta donde están los CSV de los menús
ruta = "/Workspace/Users/merlinrojasa@gmail.com/CURSO PYTHON/Proyecto/Data"

# Listo los archivos de la carpeta para confirmar que Databricks los ve bien
print(os.listdir(ruta))

In [ ]:
import pandas as pd
import glob

ruta = "/Workspace/Users/merlinrojasa@gmail.com/CURSO PYTHON/Proyecto/Data"

# glob busca todos los archivos que terminen en .csv dentro de la carpeta
archivos = glob.glob(ruta + "/*.csv")

# Leo cada CSV y los apilo en un solo DataFrame.
# ignore_index=True reinicia el índice para que no se repita entre archivos
df_menus = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos],
    ignore_index=True
)

# Vemos el resultado
df_menus.head()

In [ ]:
import pandas as pd

# Ahora cargo el maestro de restaurantes (es un único csv)
ruta = "/Workspace/Users/merlinrojasa@gmail.com/CURSO PYTHON/Proyecto/Maestros_Restaurantes/restaurants.csv"

df_restaurants = pd.read_csv(ruta)

df_restaurants.head()

### Diagnóstico inicial (Data Profiling)

In [ ]:
# Cuántas filas y columnas tiene cada DataFrame
print("Dimensiones de df_restaurants:")
print(df_restaurants.shape)

print("\nDimensiones de df_menus:")
print(df_menus.shape)

In [ ]:
# Reviso los tipos de dato de cada columna
print("Tipos de datos - df_restaurants:")
print(df_restaurants.dtypes)

print("\nTipos de datos - df_menus:")
print(df_menus.dtypes)

In [ ]:
# Cuento los nulos por columna para saber dónde hay que limpiar
print("Valores nulos - df_restaurants:")
print(df_restaurants.isnull().sum())

print("\nValores nulos - df_menus:")
print(df_menus.isnull().sum())

### TRANSFORM


In [ ]:
# Diccionario: reemplazamos los símbolos de $ a texto
mapeo_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

# .map() reemplaza cada valor y lo guardo en una columna nueva y original price_range
df_restaurants["name_rango_price"] = df_restaurants["price_range"].map(mapeo_precios)

In [ ]:
# Ver si funcionó el mapeo
df_restaurants["name_rango_price"].head()

In [ ]:
# Separo la dirección completa en 3 partes usando la coma como separador.
# n=2 hace 2 cortes, y así obtengo 3 columnas:(calle, ciudad, estado y código postal)
df_restaurants[["calle", "ciudad", "estado_zip"]] = (
    df_restaurants["full_address"]
    .str.split(",", n=2, expand=True)
)

# Quito los espacios sobrantes
df_restaurants["ciudad"] = df_restaurants["ciudad"].str.strip()

In [ ]:
# Filtro de calidad: me quedo solo con restaurantes que tengan un score válido
# (ni nulo ni cero)
df_restaurants = df_restaurants[
    df_restaurants["score"].notna() &
    (df_restaurants["score"] != 0)
]

# Estandarizo la categoría en minúsculas
df_restaurants["category"] = df_restaurants["category"].str.lower()

In [ ]:
# El precio viene como texto, por ejemplo "15,99 USD". Lo limpio en tres pasos:
df_menus["price"] = (
    df_menus["price"]
    .str.replace(" USD", "", regex=False)   # 1) quito el texto " USD"
    .str.replace(",", ".", regex=False)     # 2) cambio la coma decimal por punto
    .astype(float)                          # 3) convierto a número decimal
)

In [ ]:
df_menus.head()

In [ ]:
# Descarto los platos sin precio o con precio 0.
# .copy() creo una copia independiente de estos datos
df_menus = df_menus[
    df_menus["price"].notna() &
    (df_menus["price"] != 0.0)
].copy()

In [ ]:
# Me quedo solo con los restaurantes "consolidados" (más de 100 calificaciones)
df_restaurants = df_restaurants[
    df_restaurants["ratings"] > 100
].copy()

In [ ]:
df_restaurants.head()

In [ ]:
# Cruzo restaurantes con menús por el ID del restaurante.
# Con inner join solo quedan los restaurantes que tienen platos en el menú
df_master = pd.merge(
    df_restaurants,
    df_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner"
)

# LOAD

In [ ]:
# Instalo SQLAlchemy en el cluster de Databricks
%pip install sqlalchemy

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
# Creo el motor de conexión a una base SQLite local.
engine = create_engine("sqlite:///delivery_insights.db")

In [ ]:
# Cargo df_master como la tabla master_food_data.
# if_exists="replace" la reescribe si ya existía y index=False evita guardar
df_master.to_sql(
    "master_food_data",
    con=engine,
    if_exists="replace",
    index=False
)

## BUSINESS INTELLIGENCE

### Preguntas

**1. ¿Cuáles son las 5 ciudades con mayor cantidad de restaurantes consolidados?**
- Primero importamos `text`
- Luego hacemos la query

In [ ]:
# text() permite ejecutar SQL escrito como string con SQLAlchemy
from sqlalchemy import text

In [ ]:
# COUNT(DISTINCT id) es clave: df_master tiene una fila por plato,
# así que sin DISTINCT contaría platos y no restaurantes
query = """
SELECT
    ciudad,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY ciudad
ORDER BY cantidad_restaurantes DESC
LIMIT 5
"""

In [ ]:
# Ejecuto la query traigo el resultado
df_ciudades = pd.read_sql(text(query), engine)
df_ciudades

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# En un gráfico horizontal, matplotlib dibuja de abajo hacia arriba.
# Ordeno ascendente
df_g1 = df_ciudades.sort_values(
    "cantidad_restaurantes", ascending=True
)

fig, ax = plt.subplots(figsize=(10, 5.5))

barras = ax.barh(
    df_g1["ciudad"].astype(str),
    df_g1["cantidad_restaurantes"],
    color="#2878B5",
    height=0.65
)

# Escribo el valor al final de cada barra
ax.bar_label(barras, padding=5, fmt="%.0f", fontsize=10)
ax.set_title(
    "Penetración geográfica: Top 5 ciudades",
    fontsize=15, fontweight="bold", pad=15
)
ax.set_xlabel("Cantidad de restaurantes consolidados")
ax.set_ylabel("Ciudad")

ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Detalles de estilo
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)

# Dejo un 15% de margen a la derecha para que quepan las etiquetas
ax.set_xlim(0, df_g1["cantidad_restaurantes"].max() * 1.15)

plt.tight_layout()
plt.show()

INTERPRETACIÓN:
Milwaukee y Seattle son las ciudades con mayor cantidad de restaurantes consolidados, con 122 y 119 establecimientos, respectivamente. Ambas destacan frente a las demás ciudades analizadas.

**2. Distribución de restaurantes por rango de precios**

In [ ]:
# Cuento restaurantes (no platos) por cada rango de precio
query_p2 = """
SELECT
    name_rango_price,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY name_rango_price
ORDER BY cantidad_restaurantes DESC
"""

In [ ]:
df_precios = pd.read_sql(text(query_p2), engine)
df_precios

In [ ]:
# Gráfico de torta con el porcentaje de cada rango
plt.figure(figsize=(8, 8))

plt.pie(
    df_precios["cantidad_restaurantes"],
    labels=df_precios["name_rango_price"],
    autopct="%1.1f%%",     # muestra el porcentaje con un decimal
    startangle=90,         # empieza desde arriba
    pctdistance=0.75       # posición del porcentaje dentro de cada porción
)

plt.title("Distribución de restaurantes por rango de precios")
plt.axis("equal")          # asegura que sea un círculo y no una elipse
plt.tight_layout()
plt.show()

INTERPRETACIÓN: La mayoría de los restaurantes pertenece al rango económico, con 450 establecimientos (67,1 %), seguido de la categoría moderadamente cara, con 157 (23,4 %). Esto quiere decir que predominan las opciones económicas en el mercado.

**3. Precio promedio de los platos en las 3 ciudades más populares. ¿En qué ciudad es más caro comer?**

In [ ]:
# Uso un CTE (WITH) para primero identificar las 3 ciudades con más restaurantes,
# y luego calculo el precio promedio de los platos solo en esas ciudades
query_p3 = """
WITH top_ciudades AS (
    SELECT
        ciudad,
        COUNT(DISTINCT id) AS cantidad_restaurantes
    FROM master_food_data
    GROUP BY ciudad
    ORDER BY cantidad_restaurantes DESC
    LIMIT 3
)
SELECT
    t.ciudad,
    ROUND(AVG(m.price), 2) AS precio_promedio
FROM master_food_data AS m
INNER JOIN top_ciudades AS t
    ON m.ciudad = t.ciudad
WHERE m.price IS NOT NULL
GROUP BY t.ciudad
ORDER BY precio_promedio DESC
"""

In [ ]:
df_precios_ciudad = pd.read_sql(text(query_p3), engine)
df_precios_ciudad

In [ ]:
# Ordeno de mayor a menor precio para ver rápido cuál ciudad es la más cara
df_g3 = df_precios_ciudad.sort_values(
    "precio_promedio", ascending=False
).copy()

fig, ax = plt.subplots(figsize=(9, 5.5))

barras = ax.bar(
    df_g3["ciudad"].astype(str),
    df_g3["precio_promedio"],
    color="#269E8A",
    width=0.6
)

# Etiquetas con dos decimales sobre cada barra
ax.bar_label(
    barras,
    labels=[f"{v:.2f}" for v in df_g3["precio_promedio"]],
    padding=4,
    fontsize=10,
    fontweight="bold"
)

ax.set_title(
    "Comparación del precio promedio de los platos por ciudad",
    fontsize=14, fontweight="bold", pad=18
)
ax.set_xlabel("Ciudad")
ax.set_ylabel("Precio promedio del plato")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

# Roto un poco los nombres de las ciudades por si son largos
ax.tick_params(axis="x", rotation=20)

# Margen superior del 18% para que no se corten las etiquetas
ax.set_ylim(
    0,
    df_g3["precio_promedio"].max() * 1.18
)

plt.tight_layout()
plt.show()

INTERPRETACIÓN:Seattle presenta el precio promedio por plato más alto, con 11,98, seguida de Lynnwood con 10,82 y Milwaukee con 9,53. Esto muestra que los precios promedio varían entre las tres ciudades.

**Pregunta 4: ¿Un restaurante más caro garantiza una mejor experiencia?**

In [ ]:
# Puntaje promedio por rango de precios.
query_p4 = """
SELECT
    name_rango_price,
    ROUND(AVG(score), 2) AS puntaje_promedio
FROM (
    SELECT DISTINCT id, name_rango_price, score
    FROM master_food_data
    WHERE score IS NOT NULL
)
GROUP BY name_rango_price
ORDER BY puntaje_promedio ASC
"""

In [ ]:
df_calidad_precio = pd.read_sql(text(query_p4), engine)
df_calidad_precio

In [ ]:
df_g4 = df_calidad_precio.copy()

# Orden lógico de las categorías, de menor a mayor precio.
orden_precios = [
    "Económico",
    "Moderadamente caro",
    "Caro",
    "Muy caro"
]

#  categorías que  existen en los datos...
categorias_presentes = df_g4["name_rango_price"].dropna().tolist()
orden_final = [c for c in orden_precios if c in categorias_presentes]

# ...y agrego al final las que no estaban en mi lista, para no perder ninguna
orden_final += [c for c in categorias_presentes if c not in orden_final]

# Creo una columna auxiliar con la posición de cada categoría
df_g4["name_rango_price"] = df_g4["name_rango_price"].astype(str)
df_g4["orden"] = df_g4["name_rango_price"].map(
    {categoria: i for i, categoria in enumerate(orden_final)}
)
df_g4 = df_g4.sort_values("orden")

fig, ax = plt.subplots(figsize=(9, 5.5))

barras = ax.bar(
    df_g4["name_rango_price"],
    df_g4["puntaje_promedio"],
    color="#E59A36",
    width=0.6
)

ax.bar_label(
    barras,
    labels=[f"{v:.2f}" for v in df_g4["puntaje_promedio"]],
    padding=4,
    fontsize=10,
    fontweight="bold"
)

ax.set_title(
    "Puntaje promedio según rango de precios",
    fontsize=14, fontweight="bold", pad=18
)
ax.set_xlabel("Rango de precios (de menor a mayor)")
ax.set_ylabel("Puntaje promedio de los restaurantes")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

INTERPRETACIÓN: Los restaurantes económicos y moderadamente caros tienen puntajes promedio de 4,61 y 4,62, respectivamente, mientras que los restaurantes caros obtienen 4,43. Por lo tanto, los datos muestran que un precio más alto no garantiza una mejor calificación de los clientes.